In [17]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [5]:
# Zero line base line test

from transformers import pipeline

classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

sentence1 = "He won the match."
sentence2 = "He was sick"

premise = f"{sentence1}</s></s>{sentence2}"

result = classifier(
    premise,
    candidate_labels=["same meaning", "different meaning"],
    hypothesis_template="The two sentences have {}."
)

print(result)


Device set to use cuda:0


{'sequence': 'He won the match.</s></s>He was sick', 'labels': ['different meaning', 'same meaning'], 'scores': [0.6386207938194275, 0.3613792061805725]}


In [15]:
# : Linear Probe Baseline (embedding + small classifier)

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

model = SentenceTransformer("all-MiniLM-L6-v2")

# Example data
sent1 = ["He won the match.", "I like apples."]
sent2 = ["He was victorious in the game.", "I like oranges."]
labels = [1, 0]

# Encode sentences
X = [model.encode([s1, s2]) for s1, s2 in zip(sent1, sent2)]
X = [x[0] - x[1] for x in X]      # simple difference feature

# Train on full dataset (tiny dataset, so skip splitting)
clf = LogisticRegression().fit(X, labels)
preds = clf.predict(X)

# Evaluate on same data (smoke test)
print("F1:", f1_score(labels, preds))


F1: 1.0


In [1]:
# STEP–4: Smoke Test Fine-tuning (1 epoch, small subset)

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

dataset = load_dataset("json", data_files="data.jsonl").train_test_split(test_size=0.2)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"],
                     truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": metric.compute(predictions=preds, references=labels)["f1"]}

training_args = TrainingArguments(
    output_dir="./tmp-smoke",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


/home/bs00815/Desktop/llm-transformers-from-zero-to-pro/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: Unable to find '/home/bs00815/Desktop/llm-transformers-from-zero-to-pro/3_fine_tuning_pretrained_model/data.jsonl'